# Sun Run 2026 — extract teams data

Reads the published Teams page, builds `categories`, `teams`, and `runners`, then saves CSV files under `data/processed/`.

## Imports and paths

Libraries, source URL, and output directory.

In [19]:
import re
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

# Official results page (plain text inside HTML)
SOURCE_URL = "https://cdn-1.sportstats.one/SunRun2026_Teams.htm"

OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)


## Regex patterns

- **Category** — lines like `10K Team Results - … Category`
- **Team** — rank, total time, team name, average time in parentheses
- **Runner** — two runners per text line (rank, time, name) × 2

In [20]:
# One category block per division (e.g. Accounting Firms)
CAT = re.compile(r"^10K Team Results - (.+?) Category$")
# Team summary line after a category header
TEAM = re.compile(r"^\s*(\d+)\.\s+(\d{1,2}:\d{2}:\d{2})\s+(.+?)\s+\(\s*(\d{1,2}:\d{2}:\d{2})\)\s*$")
# Runner rows: split pairs on the line with RUN.finditer(...)
RUN = re.compile(
    r"(\d+)\s+\(?\s*(\d{1,2}:\d{2}:\d{2})\)?\s+(.+?)(?=(?:\s{2,}\d+\s+\(?\s*\d{1,2}:\d{2}:\d{2}\)?\s+)|$)"
)


def secs(t):
    """Convert H:MM:SS string to total seconds for sorting and charts."""
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s


## Load raw text

Download from `SOURCE_URL`.

In [21]:
resp = requests.get(SOURCE_URL, timeout=30)
resp.raise_for_status()
text = resp.text
if not text.strip():
    raise RuntimeError("Empty response from URL.")


## Parse line by line

State machine: current **category** → **team** header → **runner** lines until the next team or category.

In [22]:
cats, teams, runners = [], [], []
cid = tid = None  # current category_id and team_id
nc = nt = 0  # running id counters

for line in text.splitlines():
    line = line.rstrip()
    # Skip blanks and separator lines
    if not line.strip() or line.startswith("==="):
        continue

    m = CAT.match(line.strip())
    if m:
        nc += 1
        cid, tid = nc, None
        cats.append({"category_id": cid, "category_name": m.group(1).strip()})
        continue

    m = TEAM.match(line)
    if m and cid is not None:
        nt += 1
        tid = nt
        teams.append(
            {
                "team_id": tid,
                "category_id": cid,
                "category_rank": int(m.group(1)),
                "team_total_time": m.group(2),
                "team_total_seconds": secs(m.group(2)),
                "team_name": m.group(3).strip(),
                "team_avg_time": m.group(4),
                "team_avg_seconds": secs(m.group(4)),
            }
        )
        continue

    # Runner lines only make sense after a team header
    if tid is None:
        continue

    for rm in RUN.finditer(line):
        name = " ".join(rm.group(3).split())
        if not name:
            continue
        runners.append(
            {
                "team_id": tid,
                "runner_rank_in_team": int(rm.group(1)),
                "runner_time": rm.group(2),
                "runner_seconds": secs(rm.group(2)),
                "runner_name": name,
            }
        )


## Build DataFrames

Merge runner rows with team metadata for filtering by category or team name.

In [23]:
categories_df = pd.DataFrame(cats).drop_duplicates("category_id")
teams_df = pd.DataFrame(teams)
runners_df = pd.DataFrame(runners)

if len(runners_df) and len(teams_df):
    runners_df = runners_df.merge(
        teams_df[["team_id", "category_id", "team_name", "category_rank"]],
        on="team_id",
    )


## Preview

Quick sanity check of row counts and first rows.

In [24]:
print(
    len(categories_df),
    "categories",
    len(teams_df),
    "teams",
    len(runners_df),
    "runners",
)
display(categories_df.head(), teams_df.head(), runners_df.head())


29 categories 683 teams 6699 runners


,category_id,category_name
0,1,Accounting Firms
1,2,Advertising/Marketing Services
2,3,Education
3,4,Engineering
4,5,Financial - Banking & Investment


,team_id,category_id,category_rank,team_total_time,team_total_seconds,team_name,team_avg_time,team_avg_seconds
0,1,1,15,8:03:46,29026,Team Clearline,1:00:29,3629
1,2,1,16,8:14:03,29643,Smythe in the Fast Lane,1:01:46,3706
2,3,1,17,9:20:36,33636,Rolfe Benson,1:10:05,4205
3,4,1,18,10:21:49,37309,AFFINITY ACCOUNTANTS LLP,1:17:44,4664
4,5,1,19,11:46:02,42362,Loewen Kruse CPA,1:28:16,5296


,team_id,runner_rank_in_team,runner_time,runner_seconds,runner_name,category_id,team_name,category_rank
0,1,10,1:11:53,4313,Angeline Martinez,1,Team Clearline,15
1,1,11,1:17:52,4672,Annelie Vistica,1,Team Clearline,15
2,1,12,1:17:53,4673,Dan Vistica,1,Team Clearline,15
3,1,4,1:00:46,3646,Jason Ghag,1,Team Clearline,15
4,1,13,1:25:54,5154,Jason Shin,1,Team Clearline,15


## Save

Write CSV files and print each file size.

In [25]:
def _human_bytes(n: int) -> str:
    if n < 1024:
        return f"{n} B"
    if n < 1024 * 1024:
        return f"{n / 1024:.1f} KB"
    return f"{n / 1024 / 1024:.1f} MB"


for stem, df in [("categories", categories_df), ("teams", teams_df), ("runners", runners_df)]:
    path = OUT / f"sunrun_{stem}.csv"
    df.to_csv(path, index=False)
    size = path.stat().st_size
    print(f"{path.name}: {_human_bytes(size)}")

print("Saved to", OUT)


sunrun_categories.csv: 724 B
sunrun_teams.csv: 36.0 KB
sunrun_runners.csv: 365.5 KB
Saved to data/processed
